In [ ]:
import numpy as np, os, time
import tensorflow as tf
from tensorflow import keras
from argparse import Namespace
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tmodel

signal_index=2
feature_type=0
args = Namespace( signal=signal_index, feature_type=feature_type, nfeatures=14, batch_size=512, loss="mae", nstreams=10, data_dir="/explore/nobackup/projects/ilab/data/astrotime/demo", devices=[] )

In [ ]:
data=tmodel.get_demo_data()
signals = data['signals']
times = data['times']
T: np.ndarray = times[signal_index].copy()
X: np.ndarray = tmodel.get_features( T, feature_type, args )
Y: np.ndarray = signals[signal_index]

strategy = tf.distribute.MirroredStrategy([f"GPU:{i}" for i in args.devices])
print(f"Number of devices: {strategy.num_replicas_in_sync}")
with strategy.scope():
    model = tmodel.create_streams_model( X.shape[1], 0.0, n_streams=args.nstreams )
    model.compile( optimizer=tf.keras.optimizers.Adam( learning_rate=0.01 ), loss=args.loss )

latest_ckp_file = tmodel.get_ckp_file( args, "latest" )
assert os.path.exists(latest_ckp_file), f"Checkpint file '{latest_ckp_file}' not found."

In [ ]:
(At,Av) = tmodel.get_masked_attribution( model, X, Y, args )

print( At.shape )
print( Av.shape )